# KPIs (Entrega 2)

Calcula os KPIs a partir da tabela larga gerada em `preparacao_dados.ipynb` e o resultado é gravado em `src/Staging/kpis.parquet`.

- **Entrada:** `src/Staging/base_larga.parquet`.
- **Saída:** `src/Staging/kpis.parquet` 

**Convenções:** valores em R$; os sinais da base são mantidos (saídas de caixa e passivos negativos); Ano 1 = 2027 ... Ano 12 = 2038 (encerramento da concessão). As contas que não existem no Ano 12 ficam vazias, e os KPIs que dependem delas também.

## Bloco 1 - Leitura da base e identificação

O DataFrame `kpis` começa só com as colunas de identificação: `cenario` (ex.: `Cen_00001`), `cenario_num`, `ano_num` (1 a 12) e `ano_calendario` (2027 a 2038). Cada bloco seguinte acrescenta colunas de KPI a ele.

In [1]:
from pathlib import Path

import pandas as pd

PASTA_STAGING = Path("../Staging")

base = pd.read_parquet(PASTA_STAGING / "base_larga.parquet")

kpis = base[["cenario", "cenario_num", "ano_num", "ano_calendario"]].copy()

print("Base larga:", base.shape)
print("KPIs (por enquanto só identificação):", kpis.shape)

Base larga: (14400, 73)
KPIs (por enquanto só identificação): (14400, 4)


## Bloco 2 - Receita líquida, EBITDA e margem EBITDA

- `receita_liquida` = `dre_receita` + `dre_tributos`.
- `ebitda` = linha `dre_ebitda` da base da CTI
- `margem_ebitda` = `ebitda` / `receita_liquida`. 


In [3]:
kpis["receita_liquida"] = base["dre_receita"] + base["dre_tributos"]
kpis["ebitda"] = base["dre_ebitda"]
kpis["margem_ebitda"] = kpis["ebitda"] / kpis["receita_liquida"]

colunas_bloco = ["ano_calendario", "receita_liquida", "ebitda", "margem_ebitda"]
kpis[kpis["cenario"] == "Cen_00001"][colunas_bloco]

,ano_calendario,receita_liquida,ebitda,margem_ebitda
0,2027,6.251265e+09,5.188381e+09,0.829973
1,2028,5.611118e+09,4.630952e+09,0.825317
2,2029,5.721581e+09,4.883290e+09,0.853486
3,2030,5.949170e+09,5.148066e+09,0.865342
4,2031,6.255466e+09,5.453240e+09,0.871756
5,2032,6.644719e+09,5.834764e+09,0.878105
6,2033,7.053177e+09,6.236914e+09,0.884270
7,2034,7.536001e+09,6.693782e+09,0.888241
8,2035,8.000228e+09,7.136917e+09,0.892089
9,2036,8.529061e+09,7.653142e+09,0.897302


## Bloco 3 - Lucro líquido e saldo final de caixa

- `lucro_liquido` = `dre_resultado_liquido`
- `saldo_final_caixa` = `flu_saldo_final`


In [4]:
kpis["lucro_liquido"] = base["dre_resultado_liquido"]
kpis["saldo_final_caixa"] = base["flu_saldo_final"]

colunas_bloco = ["ano_calendario", "lucro_liquido", "saldo_final_caixa"]
kpis[kpis["cenario"] == "Cen_00001"][colunas_bloco]

,ano_calendario,lucro_liquido,saldo_final_caixa
0,2027,2.443243e+09,1.205265e+09
1,2028,2.000834e+09,1.022427e+09
2,2029,2.123146e+09,1.748421e+09
3,2030,2.318127e+09,1.117390e+09
4,2031,2.642606e+09,3.073864e+08
5,2032,2.955369e+09,2.661750e+08
6,2033,3.153307e+09,4.276930e+08
7,2034,3.428037e+09,1.369964e+09
8,2035,3.734760e+09,1.923333e+09
9,2036,4.238241e+09,2.325378e+09


## Bloco 4 - EBIT, capital investido e EVA

`WACC` = 10% ao ano e `ALIQUOTA_IR` = 34% (IR/CS aplicado ao EBIT).

- `ebit` = `dre_resultado_operacional`.
- `capital_investido` = -(`bal_patrimonio_liquido` + `bal_emprestimos` + `bal_emprest`)
- `eva` = `ebit` x (1 - `ALIQUOTA_IR`) - `capital_investido` x `WACC`.

In [7]:
WACC = 0.10
ALIQUOTA_IR = 0.34

patrimonio_liquido = base["bal_patrimonio_liquido"].fillna(0)
emprestimos_cp = base["bal_emprestimos"].fillna(0)
emprestimos_lp = base["bal_emprest"].fillna(0)

kpis["ebit"] = base["dre_resultado_operacional"]
kpis["capital_investido"] = -(patrimonio_liquido + emprestimos_cp + emprestimos_lp)
kpis["eva"] = kpis["ebit"] * (1 - ALIQUOTA_IR) - kpis["capital_investido"] * WACC

colunas_bloco = ["ano_calendario", "ebit", "capital_investido", "eva"]
kpis[kpis["cenario"] == "Cen_00001"][colunas_bloco]

,ano_calendario,ebit,capital_investido,eva
0,2027,4.286529e+09,7.942794e+09,2.034829e+09
1,2028,3.549559e+09,7.439164e+09,1.598792e+09
2,2029,3.709887e+09,7.533471e+09,1.695178e+09
3,2030,3.899047e+09,6.422887e+09,1.931082e+09
4,2031,4.298228e+09,5.403774e+09,2.296453e+09
5,2032,4.657110e+09,4.740097e+09,2.599683e+09
6,2033,4.913431e+09,4.044248e+09,2.838440e+09
7,2034,5.303891e+09,3.947081e+09,3.105860e+09
8,2035,5.683662e+09,3.884704e+09,3.362747e+09
9,2036,6.319144e+09,3.955801e+09,3.775055e+09


## Bloco 5 - Fluxo de caixa direto: recebimentos, pagamentos e FCO

- `dfc_recebimentos` = `flu_receita`,
- `dfc_pagamentos` = `flu_tributos` + `flu_custos` + `flu_imposto_de_renda_e_contribuicao_social`: tributos, custos e IR/CS pagos, como na fórmula do professor (FCO = Recebimentos - Tributos - Custos - IR/CS).
- `dfc_fco` = `dfc_recebimentos` + `dfc_pagamentos`.

A fórmula alternativa do professor, `Receita - Variação do CR`, é uma aproximação dos recebimentos: chega perto de `flu_receita` do Ano 5 em diante, mas diverge no início (+5% no Ano 2) e no Ano 12 (-6%), e as outras contas a receber da base não explicam a diferença. Ela entra como conferência no bloco final.

A última linha mostra o Cen_00001 para conferência visual.

In [8]:
kpis["dfc_recebimentos"] = base["flu_receita"]
kpis["dfc_pagamentos"] = (
    base["flu_tributos"]
    + base["flu_custos"]
    + base["flu_imposto_de_renda_e_contribuicao_social"]
)
kpis["dfc_fco"] = kpis["dfc_recebimentos"] + kpis["dfc_pagamentos"]

colunas_bloco = ["ano_calendario", "dfc_recebimentos", "dfc_pagamentos", "dfc_fco"]
kpis[kpis["cenario"] == "Cen_00001"][colunas_bloco]

,ano_calendario,dfc_recebimentos,dfc_pagamentos,dfc_fco
0,2027,5.396831e+09,-2.169367e+09,3.227463e+09
1,2028,5.872798e+09,-2.624139e+09,3.248659e+09
2,2029,6.123728e+09,-2.691079e+09,3.432649e+09
3,2030,6.443489e+09,-3.186545e+09,3.256944e+09
4,2031,6.812088e+09,-3.591245e+09,3.220842e+09
5,2032,7.237617e+09,-3.315415e+09,3.922202e+09
6,2033,7.710044e+09,-3.471163e+09,4.238881e+09
7,2034,8.242633e+09,-3.730966e+09,4.511667e+09
8,2035,8.742053e+09,-4.379581e+09,4.362472e+09
9,2036,9.313009e+09,-4.920874e+09,4.392134e+09


## Salvando os KPIs

Esta célula fica sempre por último e regrava `kpis.parquet` com todas as colunas calculadas nos blocos acima.

In [9]:
destino = PASTA_STAGING / "kpis.parquet"
kpis.to_parquet(destino, index=False)

print("Salvo em:", destino.resolve())
print("Formato:", kpis.shape)
print("Tamanho: %.2f MB" % (destino.stat().st_size / 1e6))

Salvo em: /Users/harryzhu/Projects/Projeto7/src/Staging/kpis.parquet
Formato: (14400, 15)
Tamanho: 1.45 MB
